In [ ]:
import pandas as pd
df=pd.read_excel("/content/RS Data for Python.xlsx")

In [ ]:
df.head()

In [ ]:
df.shape
df.columns
df.info()

In [ ]:
df.describe()
df.isnull().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

In [ ]:
import numpy as np
numerical_cols = df.select_dtypes(include=np.number).columns

for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['BirthDate'] = pd.to_datetime(df['BirthDate'])
df['JoinDate'] = pd.to_datetime(df['JoinDate'])

df['CustomerAge'] = (
    pd.Timestamp.now().year -
    df['BirthDate'].dt.year
)

df['CustomerTenure'] = (
    pd.Timestamp.now().year -
    df['JoinDate'].dt.year
)

df['SalesMonth'] = df['Date'].dt.month

In [ ]:
Q1 = df['Sales'].quantile(0.25)
Q3 = df['Sales'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df['Sales'] < lower) |
    (df['Sales'] > upper)
]

print(outliers.shape)

In [ ]:
df = df[
    (df['Sales'] >= lower) &
    (df['Sales'] <= upper)
]

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['Gender'] = encoder.fit_transform(df['Gender'])

df['PaymentMethod'] = encoder.fit_transform(
    df['PaymentMethod']
)


In [ ]:
df = pd.get_dummies(
    df,
    columns=['Category', 'Region']
)

In [ ]:
drop_cols = [
    'TransactionID',
    'FirstName',
    'LastName'
]

df = df.drop(columns=drop_cols)


In [ ]:
df['Quantity'] = df['Quantity'].astype('int16')
df['Sales'] = df['Sales'].astype('float32')
df['CustomerAge'] = df['CustomerAge'].astype('int16')
df['CustomerTenure'] = df['CustomerTenure'].astype('int16')

In [ ]:
print(df.info())
print(df.isnull().sum())
print(df.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print(df.head())

In [ ]:
total_sales = df['Sales'].sum()
print(total_sales)

total_profit = df['Profit'].sum()
print(total_profit)

In [ ]:
aov = df['Sales'].mean()
print(aov)

In [ ]:
profit_margin = (
    df['Profit'].sum() /
    df['Sales'].sum()
) * 100

print(profit_margin)

In [ ]:
category_sales = df.groupby('Category')['Sales'].sum()

print(category_sales)

In [ ]:
category_sales.plot(
    kind='bar',
    figsize=(10,5)
)

plt.title('Sales by Category')
plt.ylabel('Sales')
plt.show()

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month

In [ ]:
monthly_sales = df.groupby('Month')['Sales'].sum()
print(monthly_sales)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    monthly_sales.index,
    monthly_sales.values,
    marker='o'
)

plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Sales')

plt.show()

In [ ]:
region_sales = df.groupby('Region')['Sales'].sum()
print(region_sales)

In [ ]:
sns.barplot(
    x=region_sales.index,
    y=region_sales.values
)

plt.title('Sales by Region')

plt.show()

In [ ]:
top_customers = (
    df.groupby('CustomerID')['Sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_customers)

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    df['Sales'],
    bins=30,
    kde=True
)

plt.title('Sales Distribution')

plt.show()

In [ ]:
corr_matrix = df.corr(numeric_only=True)

print(corr_matrix)

In [ ]:
plt.figure(figsize=(12,8))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm'
)

plt.title('Correlation Matrix')

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    x='Discount',
    y='Profit',
    data=df
)

plt.title('Discount vs Profit')

plt.show()

In [ ]:
top_products = (
    df.groupby('ProductName')['Sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_products)

In [ ]:
top_products.plot(
    kind='barh',
    figsize=(10,6)
)

plt.title('Top Products')

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x=df['Sales']
)

plt.title('Sales Boxplot')

plt.show()

In [ ]:
pivot = pd.pivot_table(
    df,
    values='Profit',
    index='Category',
    columns='Region',
    aggfunc='sum'
)

print(pivot)

In [ ]:
plt.figure(figsize=(10,6))

sns.heatmap(
    pivot,
    annot=True,
    cmap='YlGnBu'
)

plt.title('Profit by Category and Region')

plt.show()

In [ ]:
fig = px.bar(
    df,
    x='Category',
    y='Sales',
    color='Region',
    title='Interactive Sales Dashboard'
)

fig.show()

In [ ]:
mean_sales = df['Sales'].mean()

print(mean_sales)

In [ ]:
median_sales = df['Sales'].median()

print(median_sales)

In [ ]:
mode_category = df['Category'].mode()

print(mode_category)

In [ ]:
variance_sales = df['Sales'].var()

print(variance_sales)

In [ ]:
std_sales = df['Sales'].std()

print(std_sales)

In [ ]:
card_prob = (
    (df['PaymentMethod'] == 'Card').sum()
    / len(df)
)

print(card_prob)

In [ ]:
sns.histplot(
    df['Sales'],
    kde=True
)

plt.title('Sales Distribution')

plt.show()

In [ ]:
sample_df = df.sample(
    n=500,
    random_state=42
)

In [ ]:
from scipy.stats import ttest_ind

male_sales = df[df['Gender']==1]['Sales']
female_sales = df[df['Gender']==0]['Sales']

t_stat, p_value = ttest_ind(
    male_sales,
    female_sales
)

print(p_value)

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['BirthDate'] = pd.to_datetime(df['BirthDate'])
df['JoinDate'] = pd.to_datetime(df['JoinDate'])

In [ ]:
df['Year'] = df['Date'].dt.year

In [ ]:
df['Month'] = df['Date'].dt.month

In [ ]:
df['Day'] = df['Date'].dt.day

In [ ]:
df['Weekday'] = df['Date'].dt.day_name()

In [ ]:
df['Quarter'] = df['Date'].dt.quarter

In [ ]:
df['CustomerAge'] = (
    pd.Timestamp.now().year -
    df['BirthDate'].dt.year
)

In [ ]:
df['CustomerTenure'] = (
    pd.Timestamp.now().year -
    df['JoinDate'].dt.year
)

In [ ]:
df['ProfitMargin'] = (
    df['Profit'] / df['Sales']
) * 100

In [ ]:
df['DiscountApplied'] = np.where(
    df['Discount'] > 0,
    1,
    0
)

In [ ]:
customer_avg_spend = (
    df.groupby('CustomerID')['Sales']
    .mean()
)

print(customer_avg_spend.head())

In [ ]:
df['AvgCustomerSpend'] = (
    df.groupby('CustomerID')['Sales']
    .transform('mean')
)

In [ ]:
df['ProductAvgSales'] = (
    df.groupby('ProductName')['Sales']
    .transform('mean')
)

In [ ]:
df['ProductTotalQuantity'] = (
    df.groupby('ProductName')['Quantity']
    .transform('sum')
)

In [ ]:
df['StoreAvgSales'] = (
    df.groupby('StoreID')['Sales']
    .transform('mean')
)

In [ ]:
df = df.sort_values('Date')

df['RollingSalesMean'] = (
    df['Sales']
    .rolling(window=7)
    .mean()
)

In [ ]:
df['PreviousSales'] = df['Sales'].shift(1)

In [ ]:
freq = df['ProductName'].value_counts()

df['ProductFreq'] = (
    df['ProductName']
    .map(freq)
)

In [ ]:
df['Discount_Profit'] = (
    df['Discount'] * df['Profit']
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scale_cols = [
    'Quantity',
    'Sales',
    'Profit',
    'CustomerAge',
    'CustomerTenure'
]

df[scale_cols] = scaler.fit_transform(
    df[scale_cols]
)

In [ ]:
corr = df.corr(numeric_only=True)

print(corr['Sales'].sort_values(
    ascending=False
))

In [ ]:
# ================================
# DAY 6 — MACHINE LEARNING FOUNDATIONS
# ================================

# ========= IMPORTS =========

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import matplotlib.pyplot as plt
import seaborn as sns

print(df.head())

# ================================
# FEATURE SELECTION
# ================================

# Selecting useful features

features = [
    'Quantity',
    'Discount',
    'UnitPrice',
    'CostPrice',
    'Profit',
    'CustomerAge',
    'CustomerTenure',
    'AvgCustomerSpend',
    'ProductAvgSales',
    'StoreAvgSales'
]

X = df[features]

# Target Variable
y = df['Sales']

# ================================
# TRAIN TEST SPLIT
# ================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

# ================================
# MODEL BUILDING
# ================================

model = LinearRegression()

# Train model
model.fit(X_train, y_train)

# ================================
# PREDICTIONS
# ================================

predictions = model.predict(X_test)

print(predictions[:5])

# ================================
# EVALUATION
# ================================

mae = mean_absolute_error(y_test, predictions)

mse = mean_squared_error(y_test, predictions)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, predictions)

print("\n========== MODEL PERFORMANCE ==========")

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

# ================================
# ACTUAL VS PREDICTED
# ================================

results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': predictions
})

print(results.head())

# ================================
# VISUALIZATION
# ================================

plt.figure(figsize=(8,5))

sns.scatterplot(
    x=y_test,
    y=predictions
)

plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")

plt.title("Actual vs Predicted Sales")

plt.show()

# ================================
# RESIDUAL ANALYSIS
# ================================

residuals = y_test - predictions

plt.figure(figsize=(8,5))

sns.histplot(
    residuals,
    kde=True
)

plt.title("Residual Distribution")

plt.show()

# ================================
# FEATURE IMPORTANCE
# ================================

coefficients = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_
})

print(coefficients)

# ================================
# MODEL INTERPRETATION
# ================================

print("\n========== INTERPRETATION ==========")

print("""
Positive coefficient:
Feature increases sales.

Negative coefficient:
Feature decreases sales.
""")

# ================================
# OVERFITTING CHECK
# ================================

train_score = model.score(X_train, y_train)

test_score = model.score(X_test, y_test)

print("\nTrain Score:", train_score)
print("Test Score :", test_score)

# ================================
# SAVING RESULTS
# ================================

results.to_csv(
    'day6_predictions.csv',
    index=False
)

print("\nDay 6 Completed Successfully")

In [ ]:
# ==========================================
# DAY 7 — MULTIPLE ML MODELS + COMPARISON
# ==========================================

# ================= IMPORTS =================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)

from sklearn.linear_model import (
    LinearRegression
)

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsRegressor

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print(df.head())

# ==========================================
# FEATURE SELECTION
# ==========================================

features = [
    'Quantity',
    'Discount',
    'UnitPrice',
    'CostPrice',
    'Profit',
    'CustomerAge',
    'CustomerTenure',
    'AvgCustomerSpend',
    'ProductAvgSales',
    'StoreAvgSales'
]

X = df[features]

y = df['Sales']

# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================================
# MODELS
# ==========================================

models = {

    'Linear Regression':
        LinearRegression(),

    'KNN':
        KNeighborsRegressor(n_neighbors=5),

    'Decision Tree':
        DecisionTreeRegressor(
            max_depth=5,
            random_state=42
        ),

    'Random Forest':
        RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            random_state=42
        )
}

# ==========================================
# POLYNOMIAL REGRESSION PIPELINE
# ==========================================

poly_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2)),
    ('linear', LinearRegression())
])

models['Polynomial Regression'] = poly_model

# ==========================================
# MODEL TRAINING + EVALUATION
# ==========================================

results = []

for name, model in models.items():

    print(f"\n========== {name} ==========")

    # Train
    model.fit(X_train, y_train)

    # Predict
    predictions = model.predict(X_test)

    # Metrics
    mae = mean_absolute_error(
        y_test,
        predictions
    )

    mse = mean_squared_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        predictions
    )

    # Cross Validation
    cv_scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='r2'
    )

    avg_cv = cv_scores.mean()

    # Save results
    results.append({
        'Model': name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2 Score': r2,
        'CV Mean': avg_cv
    })

    # Print metrics
    print("MAE :", mae)
    print("MSE :", mse)
    print("RMSE:", rmse)
    print("R2 Score:", r2)
    print("Cross Val Score:", avg_cv)

# ==========================================
# RESULTS DATAFRAME
# ==========================================

results_df = pd.DataFrame(results)

print("\n========== FINAL MODEL COMPARISON ==========\n")

print(
    results_df.sort_values(
        by='R2 Score',
        ascending=False
    )
)

# ==========================================
# BEST MODEL SELECTION
# ==========================================

best_model_name = results_df.sort_values(
    by='R2 Score',
    ascending=False
).iloc[0]['Model']

print("\nBest Model:", best_model_name)

# ==========================================
# VISUALIZE MODEL PERFORMANCE
# ==========================================

plt.figure(figsize=(12,6))

sns.barplot(
    x='Model',
    y='R2 Score',
    data=results_df
)

plt.title('Model Comparison')

plt.xticks(rotation=20)

plt.show()

# ==========================================
# RANDOM FOREST FEATURE IMPORTANCE
# ==========================================

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

print("\n========== FEATURE IMPORTANCE ==========\n")

print(importance_df)

# ==========================================
# FEATURE IMPORTANCE VISUALIZATION
# ==========================================

plt.figure(figsize=(10,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=importance_df
)

plt.title('Random Forest Feature Importance')

plt.show()

# ==========================================
# ACTUAL VS PREDICTED (BEST MODEL)
# ==========================================

best_model = models[best_model_name]

best_model.fit(X_train, y_train)

best_predictions = best_model.predict(X_test)

results_compare = pd.DataFrame({
    'Actual': y_test,
    'Predicted': best_predictions
})

print(results_compare.head())

# ==========================================
# SCATTER PLOT
# ==========================================

plt.figure(figsize=(8,5))

sns.scatterplot(
    x=y_test,
    y=best_predictions
)

plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')

plt.title(
    f'Actual vs Predicted ({best_model_name})'
)

plt.show()

# ==========================================
# RESIDUAL ANALYSIS
# ==========================================

residuals = y_test - best_predictions

plt.figure(figsize=(8,5))

sns.histplot(
    residuals,
    kde=True
)

plt.title('Residual Distribution')

plt.show()

# ==========================================
# SAVE RESULTS
# ==========================================

results_df.to_csv(
    'model_comparison_results.csv',
    index=False
)

importance_df.to_csv(
    'feature_importance.csv',
    index=False
)

print("\nDay 7 Completed Successfully")

In [ ]:
# ====================================================
# DAY 8 — ADVANCED BOOSTING + HYPERPARAMETER TUNING
# ====================================================

# ================= IMPORTS =================

# Install CatBoost if not already installed
!pip install catboost

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# BOOSTING MODELS

from xgboost import XGBRegressor

from lightgbm import LGBMRegressor

from catboost import CatBoostRegressor

# ================= LOAD DATA =================


print(df.head())

# ====================================================
# FEATURE SELECTION
# ====================================================

features = [
    'Quantity',
    'Discount',
    'UnitPrice',
    'CostPrice',
    'Profit',
    'CustomerAge',
    'CustomerTenure',
    'AvgCustomerSpend',
    'ProductAvgSales',
    'StoreAvgSales'
]

X = df[features]

y = df['Sales']

# ====================================================
# TRAIN TEST SPLIT
# ====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ====================================================
# MODEL DICTIONARY
# ====================================================

models = {

    'XGBoost': XGBRegressor(
        random_state=42
    ),

    'LightGBM': LGBMRegressor(
        random_state=42
    ),

    'CatBoost': CatBoostRegressor(
        verbose=0,
        random_state=42
    )
}

# ====================================================
# MODEL EVALUATION
# ====================================================

results = []

for name, model in models.items():

    print(f"\n========== {name} ==========")

    # Pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    predictions = pipeline.predict(X_test)

    # Metrics
    mae = mean_absolute_error(
        y_test,
        predictions
    )

    mse = mean_squared_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        predictions
    )

    # Cross Validation
    cv_scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=5,
        scoring='r2'
    )

    avg_cv = cv_scores.mean()

    # Save Results
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2 Score': r2,
        'CrossVal': avg_cv
    })

    # Print Results
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2 Score:", r2)
    print("Cross Validation:", avg_cv)

# ====================================================
# RESULTS DATAFRAME
# ====================================================

results_df = pd.DataFrame(results)

print("\n========== MODEL COMPARISON ==========\n")

print(
    results_df.sort_values(
        by='R2 Score',
        ascending=False
    )
)

# ====================================================
# BEST MODEL
# ====================================================

best_model_name = results_df.sort_values(
    by='R2 Score',
    ascending=False
).iloc[0]['Model']

print("\nBest Model:", best_model_name)

# ====================================================
# HYPERPARAMETER TUNING — XGBOOST
# ====================================================

print("\n========== HYPERPARAMETER TUNING ==========")

xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBRegressor(random_state=42))
])

param_grid = {

    'model__n_estimators': [50, 100],

    'model__max_depth': [3, 5, 7],

    'model__learning_rate': [0.01, 0.1]
}

grid_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

# Train GridSearch
grid_search.fit(X_train, y_train)

# Best Model
best_xgb = grid_search.best_estimator_

print("\nBest Parameters:")
print(grid_search.best_params_)

# ====================================================
# FINAL PREDICTIONS
# ====================================================

best_predictions = best_xgb.predict(X_test)

# ====================================================
# FINAL METRICS
# ====================================================

final_mae = mean_absolute_error(
    y_test,
    best_predictions
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        best_predictions
    )
)

final_r2 = r2_score(
    y_test,
    best_predictions
)

print("\n========== FINAL TUNED MODEL ==========")

print("Final MAE :", final_mae)
print("Final RMSE:", final_rmse)
print("Final R2  :", final_r2)

# ====================================================
# ACTUAL VS PREDICTED
# ====================================================

results_compare = pd.DataFrame({
    'Actual': y_test,
    'Predicted': best_predictions
})

print(results_compare.head())

# ====================================================
# VISUALIZATION
# ====================================================

plt.figure(figsize=(8,5))

sns.scatterplot(
    x=y_test,
    y=best_predictions
)

plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")

plt.title("Actual vs Predicted Sales")

plt.show()

# ====================================================
# RESIDUAL ANALYSIS
# ====================================================

residuals = y_test - best_predictions

plt.figure(figsize=(8,5))

sns.histplot(
    residuals,
    kde=True
)

plt.title("Residual Distribution")

plt.show()

# ====================================================
# FEATURE IMPORTANCE
# ====================================================

feature_importance = pd.DataFrame({

    'Feature': features,

    'Importance': best_xgb.named_steps[
        'model'
    ].feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print("\n========== FEATURE IMPORTANCE ==========\n")

print(feature_importance)

# ====================================================
# FEATURE IMPORTANCE VISUALIZATION
# ====================================================

plt.figure(figsize=(10,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance
)

plt.title('Feature Importance')

plt.show()

# ====================================================
# SAVE MODEL RESULTS
# ====================================================

results_df.to_csv(
    'advanced_model_results.csv',
    index=False
)

feature_importance.to_csv(
    'advanced_feature_importance.csv',
    index=False
)

# ====================================================
# SAVE BEST MODEL
# ====================================================

import joblib

joblib.dump(
    best_xgb,
    'best_sales_model.pkl'
)

print("\nBest Model Saved Successfully")

print("\nDay 8 Completed Successfully")

In [ ]:
# ====================================================
# DAY 9 — ML DEPLOYMENT + FLASK API
# ====================================================

# ================= IMPORTS =================

import pandas as pd
import numpy as np

import joblib

import logging

from flask import Flask, request, jsonify

# ====================================================
# LOGGING CONFIGURATION
# ====================================================

logging.basicConfig(
    filename='logs/model_logs.log',
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s'
)

logging.info("Application Started")

# ====================================================
# LOAD TRAINED MODEL
# ====================================================

model = joblib.load('best_sales_model.pkl')

logging.info("Model Loaded Successfully")

# ====================================================
# CREATE FLASK APP
# ====================================================

app = Flask(__name__)

# ====================================================
# HOME ROUTE
# ====================================================

@app.route('/')

def home():

    return jsonify({
        'message': 'Retail Sales Prediction API Running'
    })

# ====================================================
# PREDICTION ROUTE
# ====================================================

@app.route('/predict', methods=['POST'])

def predict():

    try:

        # ==========================================
        # GET JSON INPUT
        # ==========================================

        data = request.get_json()

        logging.info(f"Received Data: {data}")

        # ==========================================
        # CREATE DATAFRAME
        # ==========================================

        input_data = pd.DataFrame([{

            'Quantity':
                data['Quantity'],

            'Discount':
                data['Discount'],

            'UnitPrice':
                data['UnitPrice'],

            'CostPrice':
                data['CostPrice'],

            'Profit':
                data['Profit'],

            'CustomerAge':
                data['CustomerAge'],

            'CustomerTenure':
                data['CustomerTenure'],

            'AvgCustomerSpend':
                data['AvgCustomerSpend'],

            'ProductAvgSales':
                data['ProductAvgSales'],

            'StoreAvgSales':
                data['StoreAvgSales']
        }])

        # ==========================================
        # MODEL PREDICTION
        # ==========================================

        prediction = model.predict(input_data)

        predicted_sales = float(prediction[0])

        logging.info(
            f"Prediction Generated: {predicted_sales}"
        )

        # ==========================================
        # RETURN RESPONSE
        # ==========================================

        return jsonify({

            'PredictedSales':
                predicted_sales,

            'Status':
                'Success'
        })

    except Exception as e:

        logging.error(f"Prediction Error: {str(e)}")

        return jsonify({

            'Status': 'Error',

            'Message': str(e)

        })

# ====================================================
# RUN APPLICATION
# ====================================================

if __name__ == '__main__':

    app.run(
        host='0.0.0.0',
        port=5000,
        debug=True
    )

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


In [ ]:
# =========================================================
# DAY 10 — FINAL CAPSTONE PROJECT
# =========================================================

# ================= IMPORTS =================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import joblib

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

# =========================================================
# LOAD DATA
# =========================================================



print(df.head())

# =========================================================
# BUSINESS KPIs
# =========================================================

total_sales = df['Sales'].sum()

total_profit = df['Profit'].sum()

avg_order_value = df['Sales'].mean()

profit_margin = (
    total_profit / total_sales
) * 100

print("\n========== BUSINESS KPIs ==========")

print("Total Sales:", total_sales)

print("Total Profit:", total_profit)

print("Average Order Value:", avg_order_value)

print("Profit Margin:", profit_margin)

# =========================================================
# FEATURE SELECTION
# =========================================================

features = [

    'Quantity',
    'Discount',
    'UnitPrice',
    'CostPrice',
    'Profit',
    'CustomerAge',
    'CustomerTenure',
    'AvgCustomerSpend',
    'ProductAvgSales',
    'StoreAvgSales'
]

X = df[features]

y = df['Sales']

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =========================================================
# PIPELINE
# =========================================================

pipeline = Pipeline([

    ('scaler', StandardScaler()),

    ('model', XGBRegressor(
        random_state=42
    ))
])

# =========================================================
# HYPERPARAMETER TUNING
# =========================================================

param_grid = {

    'model__n_estimators': [100],

    'model__max_depth': [5, 7],

    'model__learning_rate': [0.05, 0.1]
}

grid_search = GridSearchCV(

    estimator=pipeline,

    param_grid=param_grid,

    cv=3,

    scoring='r2',

    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("\nBest Parameters:")

print(grid_search.best_params_)

# =========================================================
# PREDICTIONS
# =========================================================

predictions = best_model.predict(X_test)

# =========================================================
# EVALUATION
# =========================================================

mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions)
)

r2 = r2_score(
    y_test,
    predictions
)

print("\n========== MODEL PERFORMANCE ==========")

print("MAE :", mae)

print("RMSE:", rmse)

print("R2 Score:", r2)

# =========================================================
# FEATURE IMPORTANCE
# =========================================================

importance_df = pd.DataFrame({

    'Feature': features,

    'Importance': best_model.named_steps[
        'model'
    ].feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

print("\n========== FEATURE IMPORTANCE ==========")

print(importance_df)

# =========================================================
# VISUALIZATIONS
# =========================================================

# ---------- ACTUAL VS PREDICTED ----------

plt.figure(figsize=(8,5))

sns.scatterplot(
    x=y_test,
    y=predictions
)

plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")

plt.title("Actual vs Predicted Sales")

plt.show()

# ---------- FEATURE IMPORTANCE ----------

plt.figure(figsize=(10,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=importance_df
)

plt.title("Feature Importance")

plt.show()

# ---------- MONTHLY SALES ----------

df['Date'] = pd.to_datetime(df['Date'])

df['Month'] = df['Date'].dt.month

monthly_sales = df.groupby('Month')[
    'Sales'
].sum()

plt.figure(figsize=(10,5))

monthly_sales.plot(marker='o')

plt.title("Monthly Sales Trend")

plt.xlabel("Month")

plt.ylabel("Sales")

plt.show()

# =========================================================
# INTERACTIVE DASHBOARD VISUALIZATION
# =========================================================

fig = px.bar(

    df,

    x='Category',

    y='Sales',

    color='Region',

    title='Sales by Category and Region'
)

fig.show()

# =========================================================
# SAVE MODEL
# =========================================================

joblib.dump(
    best_model,
    'final_sales_model.pkl'
)

print("\nFinal Model Saved Successfully")

# =========================================================
# SAVE RESULTS
# =========================================================

results_df = pd.DataFrame({

    'Actual': y_test,

    'Predicted': predictions
})

results_df.to_csv(
    'final_predictions.csv',
    index=False
)

importance_df.to_csv(
    'final_feature_importance.csv',
    index=False
)

print("\nResults Saved Successfully")

# =========================================================
# FINAL BUSINESS INSIGHTS
# =========================================================

print("\n========== FINAL BUSINESS INSIGHTS ==========")

print("""

1. Strongest sales drivers identified.

2. Seasonal sales trends detected.

3. Customer behavior patterns analyzed.

4. Product performance evaluated.

5. Regional performance compared.

6. ML model successfully predicts sales.

7. Boosting model outperformed baseline models.

8. Feature engineering improved performance significantly.

""")

print("\nDay 10 Capstone Completed Successfully")